In [1]:
!pip install anndata==0.8.0

In [1]:
from samap.mapping import SAMAP
from samap.analysis import (get_mapping_scores, GenePairFinder, transfer_annotations,
                            sankey_plot, chord_plot, CellTypeTriangles, 
                            ParalogSubstitutions, FunctionalEnrichment,
                            convert_eggnog_to_homologs, GeneTriangles)
from samalg import SAM
import pandas as pd
from Bio import SeqIO
from samap.utils import (save_samap, load_samap)
import scanpy as sc
import matplotlib.colors
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from scipy import sparse 
from scipy import cluster
import seaborn as sns
import random
import sklearn
from sklearn.neighbors import KernelDensity
import time
import dill
import pickle
from scipy.spatial import distance
import os
import scipy

In [2]:
with open('../../parent_dict_v2.pkl', 'rb') as f:
    parent_dict = pickle.load(f)

In [69]:
fn = '../../Active_SAM_joined/SAM_CJ_joined_v2_cleaned_03122025.h5ad'

In [70]:
sam = SAM()
sam.load_data(fn)

In [71]:
lsaturn_mv = pd.read_csv('../../SATURN_hypo_september2026_mapping_30_seeds/mouse_vole_quail_30_seeds.csv', index_col = 'Unnamed: 0')
lsaturn_m = pd.read_csv('../../SATURN_hypo_september2026_mapping_30_seeds/mouse_quail_30_seeds.csv', index_col = 'barcode')

In [72]:
ind = pd.read_csv('../../Active_SAMap_Joined/CJ_MG_metadata_08312026/CJ_MG_mapping_cleaned_08312026_0.csv')['Unnamed: 0']
lsamap_m = pd.DataFrame(index = list(ind))
for i in range(30):
    df = pd.read_csv('../../Active_SAMap_Joined/CJ_MG_metadata_08312026/CJ_MG_mapping_cleaned_08312026_'+str(i)+'.csv', index_col = 'Unnamed: 0')
    lsamap_m = pd.concat([lsamap_m, df], axis = 1)

In [73]:
meta_folder = '../../Active_SAMap_Joined/CJ_metadata_08312026/'

In [74]:
mn = os.listdir('../../Active_SAMap_Joined/CJ_metadata_08312026/')

In [75]:
pd.read_csv(meta_folder + mn[3])

,Unnamed: 0,orig.ident,nCount_RNA,nFeature_RNA,n_genes,n_counts,key,leiden_clusters,subclass_id_label_mapping,subclass_id_label_lc
0,AAACCCAAGACGACTG Run 11 sample 8,CJ,1455.0,797,797,1455.0,Run 11 sample 8,170,085 SI-MPO-LPO Lhx8 Gaba,170
1,AAACCCAAGTGCTCAT Run 11 sample 8,CJ,1576.0,728,728,1576.0,Run 11 sample 8,81,327 Oligo NN,81
2,AAACCCAGTCGAAACG Run 11 sample 8,CJ,1489.0,763,763,1489.0,Run 11 sample 8,23,128 VMH Fezf1 Glut,23
3,AAACCCAGTCTTGAAC Run 11 sample 8,CJ,797.0,521,521,797.0,Run 11 sample 8,292,327 Oligo NN,292
4,AAACCCAGTGCCGGTT Run 11 sample 8,CJ,1874.0,951,951,1874.0,Run 11 sample 8,84,090 BST-MPN Six3 Nrgn Gaba,84
...,...,...,...,...,...,...,...,...,...,...
74753,TTTGGTTTCCACAGCG Run 11 sample 7,CJ,6076.0,2117,2117,6076.0,Run 11 sample 7,135,125 DMH Hmx2 Glut,135
74754,TTTGGTTTCCATCAGA Run 11 sample 7,CJ,13270.0,3222,3222,13270.0,Run 11 sample 7,72,Unlabeled,72
74755,TTTGGTTTCGGTGCAC Run 11 sample 7,CJ,7299.0,2248,2248,7299.0,Run 11 sample 7,174,129 VMH Nr5a1 Glut,174
74756,TTTGTTGAGGAATTAC Run 11 sample 7,CJ,8517.0,2544,2544,8517.0,Run 11 sample 7,67,057 NDB-SI-MA-STRv Lhx8 Gaba,67


In [76]:
test = pd.read_csv(meta_folder + mn[0])['subclass_id_label_mapping']

In [77]:
barcodes = pd.read_csv(meta_folder + mn[0])['Unnamed: 0']

In [78]:
raw_lc = pd.read_csv(meta_folder + mn[0])['subclass_id_label_lc']

In [79]:
df_lc = pd.DataFrame(data = list(raw_lc), index = list(barcodes), columns = ['raw_lc'])

In [80]:
mode_df = pd.DataFrame(index = [a for a in range(len(test))], columns = [b for b in range(len(mn))])
for j in range(len(mn)):
    print(mn[j])
    dat = list(pd.read_csv(meta_folder + mn[j])['subclass_id_label_lc'])
    mode_df.loc[:,j] = dat

CJ_metadata_subclass_250_10_08312026.csv
CJ_metadata_subclass_250_1_08312026.csv
CJ_metadata_subclass_250_23_08312026.csv
CJ_metadata_subclass_250_8_08312026.csv
CJ_metadata_subclass_250_12_08312026.csv
CJ_metadata_subclass_250_2_08312026.csv
CJ_metadata_subclass_250_19_08312026.csv
CJ_metadata_subclass_250_21_08312026.csv
CJ_metadata_subclass_250_16_08312026.csv
CJ_metadata_subclass_250_9_08312026.csv
CJ_metadata_subclass_250_15_08312026.csv
CJ_metadata_subclass_250_22_08312026.csv
CJ_metadata_subclass_250_3_08312026.csv
CJ_metadata_subclass_250_11_08312026.csv
CJ_metadata_subclass_250_28_08312026.csv
CJ_metadata_subclass_250_27_08312026.csv
CJ_metadata_subclass_250_26_08312026.csv
CJ_metadata_subclass_250_13_08312026.csv
CJ_metadata_subclass_250_14_08312026.csv
CJ_metadata_subclass_250_6_08312026.csv
CJ_metadata_subclass_250_5_08312026.csv
CJ_metadata_subclass_250_7_08312026.csv
CJ_metadata_subclass_250_20_08312026.csv
CJ_metadata_subclass_250_29_08312026.csv
CJ_metadata_subclass_250

In [81]:
fin = []
for item in mode_df.columns:
    fin.append(mode_df.loc[10000,item])

In [82]:
scipy.stats.mode(fin)

ModeResult(mode=array([304]), count=array([30]))

In [83]:
import re

mn = sorted(mn, key=lambda s: [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', s)])

In [84]:
mn

['CJ_metadata_subclass_250_0_08312026.csv',
 'CJ_metadata_subclass_250_1_08312026.csv',
 'CJ_metadata_subclass_250_2_08312026.csv',
 'CJ_metadata_subclass_250_3_08312026.csv',
 'CJ_metadata_subclass_250_4_08312026.csv',
 'CJ_metadata_subclass_250_5_08312026.csv',
 'CJ_metadata_subclass_250_6_08312026.csv',
 'CJ_metadata_subclass_250_7_08312026.csv',
 'CJ_metadata_subclass_250_8_08312026.csv',
 'CJ_metadata_subclass_250_9_08312026.csv',
 'CJ_metadata_subclass_250_10_08312026.csv',
 'CJ_metadata_subclass_250_11_08312026.csv',
 'CJ_metadata_subclass_250_12_08312026.csv',
 'CJ_metadata_subclass_250_13_08312026.csv',
 'CJ_metadata_subclass_250_14_08312026.csv',
 'CJ_metadata_subclass_250_15_08312026.csv',
 'CJ_metadata_subclass_250_16_08312026.csv',
 'CJ_metadata_subclass_250_17_08312026.csv',
 'CJ_metadata_subclass_250_18_08312026.csv',
 'CJ_metadata_subclass_250_19_08312026.csv',
 'CJ_metadata_subclass_250_20_08312026.csv',
 'CJ_metadata_subclass_250_21_08312026.csv',
 'CJ_metadata_subcla

In [85]:
lsamap_mv = pd.DataFrame(index = [a for a in barcodes], columns = [b for b in range(len(mn))])
for j in range(len(mn)):
    dat = list(pd.read_csv(meta_folder + mn[j])['subclass_id_label_mapping'])
    lsamap_mv.loc[:,j] = dat

In [86]:
#for quail updated
threshold = 0
a = 0
b = 0
saturn_barcodes = []
saturn_barcodes_samap_mouse = []
samap_barcodes = []
samap_barcodes_saturn_mouse = []
mapping_dict ={}
for lc in sam.adata.obs['eq_subclass_lc'].unique():
    inp = ['Unlabeled', 'Unlabeled', 'Unlabeled']
    barcodes = sam.adata.obs_names[sam.adata.obs['eq_subclass_lc'] == lc]
    ct_saturn_mv = lsaturn_mv[lsaturn_mv.index.isin(barcodes)].values.ravel()
    ct_saturn_m = lsaturn_m[lsaturn_m.index.isin(barcodes)].values.ravel()
    ct_samap_mv = lsamap_mv[lsamap_mv.index.isin(barcodes)].values.ravel()
    ct_samap_m = lsamap_m[lsamap_m.index.isin(barcodes)].values.ravel()
    
    mode_saturn_mv,f_saturn_mv = scipy.stats.mode(ct_saturn_mv)
    mode_saturn_m,f_saturn_m = scipy.stats.mode(ct_saturn_m)
    mode_samap_mv,f_samap_mv = scipy.stats.mode(ct_samap_mv)
    mode_samap_m,f_samap_m = scipy.stats.mode(ct_samap_m)
    
    inp[0] = mode_saturn_mv[0]
    inp[1] = mode_samap_mv[0]
    if inp[0] == inp[1]:
        a += len(barcodes)
    if inp[0] != inp[1] and inp[0] == 'Unlabeled' and mode_saturn_m[0] != inp[1]:
        samap_barcodes.extend(list(barcodes))
    if inp[0] != inp[1] and inp[0] == 'Unlabeled' and mode_saturn_m[0] == inp[1]:
        samap_barcodes_saturn_mouse.extend(list(barcodes))
    if inp[0] != inp[1] and inp[1] == 'Unlabeled' and mode_samap_m[0] !=inp[0]:
        saturn_barcodes.extend(list(barcodes))
    if inp[0] != inp[1] and inp[1] == 'Unlabeled' and mode_samap_m[0] ==inp[0]:
        saturn_barcodes_samap_mouse.extend(list(barcodes))
    if mode_samap_m[0] == mode_saturn_m[0]:
        inp[2] = mode_samap_m[0]
    if inp[0] != inp[1] and inp[0] != 'Unlabeled' and inp[1] != 'Unlabeled':
        if inp[0] != inp[1]:
            if inp[0] == '136 PMv-TMv Pitx2 Glut':
                inp = ['cj_m136_m138']
            if inp[0] == '099 SBPV-PVa Six6 Satb2 Gaba':
                inp = ['cj_m091_m099']
        b += len(barcodes)
    inp_val = list(set(inp) - set(['Unlabeled']))
    if len(inp_val) >1:
        print(inp)
    if len(inp_val) == 0:
        mapping_dict[lc] = 'Unlabeled'
    elif len(inp_val) == 1:
        mapping_dict[lc] = inp_val[0]

In [21]:
(a + b + len(samap_barcodes) + len(saturn_barcodes) + len(saturn_barcodes_samap_mouse) + len(samap_barcodes_saturn_mouse))/len(sam.adata)

1.0

In [21]:
mismatch = b/len(sam.adata)
print(mismatch)

0.010741325343107093


In [25]:
match = a/len(sam.adata)
print(match)

0.6655073704486476


In [26]:
len(samap_barcodes)/len(sam.adata)

0.14813130367318547

In [27]:
len(samap_barcodes_saturn_mouse)/len(sam.adata)

0.1453356162551165

In [28]:
len(saturn_barcodes)/len(sam.adata)

0.003504641643703684

In [29]:
len(saturn_barcodes_samap_mouse)/len(sam.adata)

0.0267797426362396

In [31]:
samap_mgmo_saturn_mg = (len(samap_barcodes_saturn_mouse)/len(sam.adata))/(len(samap_barcodes)/len(sam.adata) + len(samap_barcodes_saturn_mouse)/len(sam.adata))
samap_mgmo_saturn_mg

0.49523679292583983

In [32]:
subset = sam.adata[sam.adata.obs_names.isin(samap_barcodes)]

In [65]:
#for anole updated
threshold = 0
a = 0
b = 0
saturn_barcodes = []
saturn_barcodes_samap_mouse = []
samap_barcodes = []
samap_barcodes_saturn_mouse = []
mapping_dict ={}
for lc in sam.adata.obs['eq_subclass_lc'].unique():
    inp = ['Unlabeled', 'Unlabeled', 'Unlabeled']
    barcodes = sam.adata.obs_names[sam.adata.obs['eq_subclass_lc'] == lc]
    ct_saturn_mv = lsaturn_mv[lsaturn_mv.index.isin(barcodes)].values.ravel()
    ct_saturn_m = lsaturn_m[lsaturn_m.index.isin(barcodes)].values.ravel()
    ct_samap_mv = lsamap_mv[lsamap_mv.index.isin(barcodes)].values.ravel()
    ct_samap_m = lsamap_m[lsamap_m.index.isin(barcodes)].values.ravel()
    
    mode_saturn_mv,f_saturn_mv = scipy.stats.mode(ct_saturn_mv)
    mode_saturn_m,f_saturn_m = scipy.stats.mode(ct_saturn_m)
    mode_samap_mv,f_samap_mv = scipy.stats.mode(ct_samap_mv)
    mode_samap_m,f_samap_m = scipy.stats.mode(ct_samap_m)
    
    inp[0] = mode_saturn_mv[0]
    inp[1] = mode_samap_mv[0]
    if inp[0] == inp[1]:
        a += len(barcodes)
    if inp[0] != inp[1] and inp[0] == 'Unlabeled' and mode_saturn_m[0] != inp[1]:
        samap_barcodes.extend(list(barcodes))
    if inp[0] != inp[1] and inp[0] == 'Unlabeled' and mode_saturn_m[0] == inp[1]:
        samap_barcodes_saturn_mouse.extend(list(barcodes))
    if inp[0] != inp[1] and inp[1] == 'Unlabeled' and mode_samap_m[0] !=inp[0]:
        saturn_barcodes.extend(list(barcodes))
    if inp[0] != inp[1] and inp[1] == 'Unlabeled' and mode_samap_m[0] ==inp[0]:
        saturn_barcodes_samap_mouse.extend(list(barcodes))
    if mode_samap_m[0] == mode_saturn_m[0]:
        inp[2] = mode_samap_m[0]
    if inp[0] != inp[1] and inp[0] != 'Unlabeled' and inp[1] != 'Unlabeled':
        if inp[0] == '136 PMv-TMv Pitx2 Glut':
            inp = ['ac_m136_m138']
        if inp[0] == '086 MPO-ADP Lhx8 Gaba':
            inp = ['ac_m058_m086']
        b += len(sam.adata.obs_names[sam.adata.obs['eq_subclass_lc'] == lc])
    inp_val = list(set(inp) - set(['Unlabeled']))
    if len(inp_val) >1:
        print(inp)
    if len(inp_val) == 0:
        mapping_dict[lc] = 'Unlabeled'
    elif len(inp_val) == 1:
        mapping_dict[lc] = inp_val[0]

In [21]:
(a + b + len(samap_barcodes) + len(saturn_barcodes) + len(saturn_barcodes_samap_mouse) + len(samap_barcodes_saturn_mouse))/len(sam.adata)

1.0

In [22]:
mismatch = b/len(sam.adata)
print(mismatch)

0.018914405010438413


In [23]:
match = a/len(sam.adata)
print(match)

0.7030062630480167


In [24]:
len(samap_barcodes)/len(sam.adata)

0.12235908141962422

In [25]:
len(samap_barcodes_saturn_mouse)/len(sam.adata)

0.15572025052192068

In [26]:
len(saturn_barcodes)/len(sam.adata)

0.0

In [27]:
len(saturn_barcodes_samap_mouse)/len(sam.adata)

0.0

In [28]:
samap_mgmo_saturn_mg = (len(samap_barcodes_saturn_mouse)/len(sam.adata))/(len(samap_barcodes)/len(sam.adata) + len(samap_barcodes_saturn_mouse)/len(sam.adata))
samap_mgmo_saturn_mg

0.5599849849849851

In [29]:
subset = sam.adata[sam.adata.obs_names.isin(samap_barcodes)]

In [19]:
#for xenopus updated
threshold = 0
a = 0
b = 0
saturn_barcodes = []
saturn_barcodes_samap_mouse = []
samap_barcodes = []
samap_barcodes_saturn_mouse = []
mapping_dict ={}
for lc in sam.adata.obs['eq_subclass_lc'].unique():
    inp = ['Unlabeled', 'Unlabeled', 'Unlabeled']
    barcodes = sam.adata.obs_names[sam.adata.obs['eq_subclass_lc'] == lc]
    ct_saturn_mv = lsaturn_mv[lsaturn_mv.index.isin(barcodes)].values.ravel()
    ct_saturn_m = lsaturn_m[lsaturn_m.index.isin(barcodes)].values.ravel()
    ct_samap_mv = lsamap_mv[lsamap_mv.index.isin(barcodes)].values.ravel()
    ct_samap_m = lsamap_m[lsamap_m.index.isin(barcodes)].values.ravel()
    
    mode_saturn_mv,f_saturn_mv = scipy.stats.mode(ct_saturn_mv)
    mode_saturn_m,f_saturn_m = scipy.stats.mode(ct_saturn_m)
    mode_samap_mv,f_samap_mv = scipy.stats.mode(ct_samap_mv)
    mode_samap_m,f_samap_m = scipy.stats.mode(ct_samap_m)
    
    inp[0] = mode_saturn_mv[0]
    inp[1] = mode_samap_mv[0]
    if inp[0] == inp[1]:
        a += len(barcodes)
    if inp[0] != inp[1] and inp[0] == 'Unlabeled' and mode_saturn_m[0] != inp[1]:
        samap_barcodes.extend(list(barcodes))
    if inp[0] != inp[1] and inp[0] == 'Unlabeled' and mode_saturn_m[0] == inp[1]:
        samap_barcodes_saturn_mouse.extend(list(barcodes))
    if inp[0] != inp[1] and inp[1] == 'Unlabeled' and mode_samap_m[0] !=inp[0]:
        saturn_barcodes.extend(list(barcodes))
    if inp[0] != inp[1] and inp[1] == 'Unlabeled' and mode_samap_m[0] ==inp[0]:
        saturn_barcodes_samap_mouse.extend(list(barcodes))
    if mode_samap_m[0] == mode_saturn_m[0]:
        inp[2] = mode_samap_m[0]
    if inp[0] != inp[1] and inp[0] != 'Unlabeled' and inp[1] != 'Unlabeled':
        if inp[0] == '136 PMv-TMv Pitx2 Glut':
            inp = ['xt_m136_m138']
        b += len(sam.adata.obs_names[sam.adata.obs['eq_subclass_lc'] == lc])
    inp_val = list(set(inp) - set(['Unlabeled']))
    if len(inp_val) >1:
        print(inp)
    if len(inp_val) == 0:
        mapping_dict[lc] = 'Unlabeled'
    elif len(inp_val) == 1:
        mapping_dict[lc] = inp_val[0]

In [21]:
(a + b + len(samap_barcodes) + len(saturn_barcodes) + len(saturn_barcodes_samap_mouse) + len(samap_barcodes_saturn_mouse))/len(sam.adata)

1.0

In [22]:
mismatch = b/len(sam.adata)
print(mismatch)

0.005258046943464153


In [23]:
match = a/len(sam.adata)
print(match)

0.6100992397148338


In [24]:
len(samap_barcodes)/len(sam.adata)

0.0831576703536155

In [25]:
len(samap_barcodes_saturn_mouse)/len(sam.adata)

0.2873214750953317

In [26]:
len(saturn_barcodes)/len(sam.adata)

0.008502877714881221

In [27]:
len(saturn_barcodes_samap_mouse)/len(sam.adata)

0.00566069017787357

In [28]:
samap_mgmo_saturn_mg = (len(samap_barcodes_saturn_mouse)/len(sam.adata))/(len(samap_barcodes)/len(sam.adata) + len(samap_barcodes_saturn_mouse)/len(sam.adata))
samap_mgmo_saturn_mg

0.775540212249073

In [29]:
subset = sam.adata[sam.adata.obs_names.isin(samap_barcodes)]

In [41]:
#for zebrafish updated
threshold = 0
a = 0
b = 0
saturn_barcodes = []
saturn_barcodes_samap_mouse = []
samap_barcodes = []
samap_barcodes_saturn_mouse = []
mapping_dict ={}
for lc in sam.adata.obs['eq_subclass_lc'].unique():
    inp = ['Unlabeled', 'Unlabeled', 'Unlabeled']
    barcodes = sam.adata.obs_names[sam.adata.obs['eq_subclass_lc'] == lc]
    ct_saturn_mv = lsaturn_mv[lsaturn_mv.index.isin(barcodes)].values.ravel()
    ct_saturn_m = lsaturn_m[lsaturn_m.index.isin(barcodes)].values.ravel()
    ct_samap_mv = lsamap_mv[lsamap_mv.index.isin(barcodes)].values.ravel()
    ct_samap_m = lsamap_m[lsamap_m.index.isin(barcodes)].values.ravel()
    
    mode_saturn_mv,f_saturn_mv = scipy.stats.mode(ct_saturn_mv)
    mode_saturn_m,f_saturn_m = scipy.stats.mode(ct_saturn_m)
    mode_samap_mv,f_samap_mv = scipy.stats.mode(ct_samap_mv)
    mode_samap_m,f_samap_m = scipy.stats.mode(ct_samap_m)
    
    inp[0] = mode_saturn_mv[0]
    inp[1] = mode_samap_mv[0]
    if inp[0] == inp[1]:
        a += len(barcodes)
    if inp[0] != inp[1] and inp[0] == 'Unlabeled' and mode_saturn_m[0] != inp[1]:
        samap_barcodes.extend(list(barcodes))
    if inp[0] != inp[1] and inp[0] == 'Unlabeled' and mode_saturn_m[0] == inp[1]:
        samap_barcodes_saturn_mouse.extend(list(barcodes))
    if inp[0] != inp[1] and inp[1] == 'Unlabeled' and mode_samap_m[0] !=inp[0]:
        saturn_barcodes.extend(list(barcodes))
    if inp[0] != inp[1] and inp[1] == 'Unlabeled' and mode_samap_m[0] ==inp[0]:
        saturn_barcodes_samap_mouse.extend(list(barcodes))
    if mode_samap_m[0] == mode_saturn_m[0]:
        inp[2] = mode_samap_m[0]
    if inp[1] != inp[2] and inp[1] != 'Unlabeled' and inp[2] != 'Unlabeled':
        print(inp[0],inp[1],inp[2],lc)
        b += len(sam.adata.obs_names[sam.adata.obs['eq_subclass_lc'] == lc])
    inp_val = list(set(inp) - set(['Unlabeled']))
    if len(inp_val) >1:
        print(inp)
    if len(inp_val) == 0:
        mapping_dict[lc] = 'Unlabeled'
    elif len(inp_val) == 1:
        mapping_dict[lc] = inp_val[0]

In [53]:
(a + b + len(samap_barcodes) + len(saturn_barcodes) + len(saturn_barcodes_samap_mouse) + len(samap_barcodes_saturn_mouse))/len(sam.adata)

1.0

In [54]:
mismatch = b/len(sam.adata)
print(mismatch)

0.0


In [55]:
match = a/len(sam.adata)
print(match)

0.9252156926583103


In [56]:
len(samap_barcodes)/len(sam.adata)

0.04558033534103858

In [57]:
len(samap_barcodes_saturn_mouse)/len(sam.adata)

0.029203972000651147

In [58]:
len(saturn_barcodes)/len(sam.adata)

0.0

In [59]:
len(saturn_barcodes_samap_mouse)/len(sam.adata)

0.0

In [60]:
samap_mgmo_saturn_mg = (len(samap_barcodes_saturn_mouse)/len(sam.adata))/(len(samap_barcodes)/len(sam.adata) + len(samap_barcodes_saturn_mouse)/len(sam.adata))
samap_mgmo_saturn_mg

0.39050936003482806

In [87]:
new_mapping = []
new_mapping_name = 'test'
for item in sam.adata.obs['eq_subclass_lc']:
    new_mapping.append(mapping_dict[item])
sam.adata.obs[new_mapping_name] = new_mapping

In [88]:
a = 0
fin_obs = []
for item in sam.adata.obs_names:
    if sam.adata.obs.loc[item,'ss_subclass'] != sam.adata.obs.loc[item,'test']:
        a += 1
        fin_obs.append(item)

In [89]:
a

0

In [33]:
a = 0 
for item in subset.obs['ss_subclass'].unique():
    if parent_dict[parent_dict[item]] == 'hypo':
        if len(subset[subset.obs['ss_subclass'] == item]) == len(sam.adata.obs[sam.adata.obs['ss_subclass'] == item]):
            a += 1
            print(item)

107 DMH Hmx2 Gaba
140 PMd-LHA Foxb1 Glut
105 TMd-DMH Foxd2 Gaba
073 MEA-BST Sox6 Gaba
143 MM-ant Foxb1 Glut
091 ARH-PVi Six6 Dopa-Gaba


In [34]:
a

6

In [35]:
sam.save_anndata(fn)